In [2]:
# Untuk gambar dan array
import os
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
import base64

# Untuk ECC encryption (ECIES)
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

# Untuk deep learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, concatenate
from tensorflow.keras.optimizers import Adam

# Untuk hashing
import hashlib

In [1]:
# Konstanta
IMG_SIZE = (64, 64)
NUM_IMAGES = 1000

In [3]:
# Load CSV dan ambil gambar
df = pd.read_csv("results.csv", delimiter='|')
df

,image_name,comment_number,comment
0,1000092795.jpg,0,Two young guys with shaggy hair look at their...
1,1000092795.jpg,1,"Two young , White males are outside near many..."
2,1000092795.jpg,2,Two men in green shirts are standing in a yard .
3,1000092795.jpg,3,A man in a blue shirt standing in a garden .
4,1000092795.jpg,4,Two friends enjoy time spent together .
...,...,...,...
158910,998845445.jpg,0,A man in shorts and a Hawaiian shirt leans ov...
158911,998845445.jpg,1,"A young man hanging over the side of a boat ,..."
158912,998845445.jpg,2,A man is leaning off of the side of a blue an...
158913,998845445.jpg,3,"A man riding a small boat in a harbor , with ..."


In [ ]:
# Load CSV dan ambil gambar
df = pd.read_csv("results.csv", delimiter='|')
comments = df['comment'].dropna().head(NUM_IMAGES)

# Fungsi konversi teks ke gambar (dummy grayscale untuk secret image)
def text_to_image(text):
    text_bytes = text.encode('utf-8')
    img_array = np.frombuffer(text_bytes, dtype=np.uint8)
    size = int(np.ceil(np.sqrt(len(img_array))))
    padded = np.pad(img_array, (0, size*size - len(img_array)))
    return padded.reshape((size, size)).astype(np.uint8)

# Preprocessing
cover_images = []
secret_images = []

for text in comments:
    # Dummy image (misalnya semua cover image putih)
    cover = Image.fromarray(np.ones((64, 64, 3), dtype=np.uint8) * 255)
    cover = cover.resize(IMG_SIZE).convert('RGB')
    cover = np.array(cover) / 255.0
    
    # Secret image dari teks
    secret_img = text_to_image(text)
    secret_img = Image.fromarray(secret_img).resize(IMG_SIZE)
    secret_img = np.stack([np.array(secret_img)]*3, axis=-1) / 255.0
    
    cover_images.append(cover)
    secret_images.append(secret_img)

cover_images = np.array(cover_images)
secret_images = np.array(secret_images)


In [ ]:
def generate_keys():
    private_key = ec.generate_private_key(ec.SECP256R1(), default_backend())
    public_key = private_key.public_key()
    return private_key, public_key

def derive_shared_key(private_key, peer_public_key):
    shared_key = private_key.exchange(ec.ECDH(), peer_public_key)
    derived_key = HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                       info=b'handshake data', backend=default_backend()).derive(shared_key)
    return derived_key

def encrypt_data(data, key):
    iv = os.urandom(16)
    cipher = Cipher(algorithms.AES(key), modes.CFB(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    ct = encryptor.update(data) + encryptor.finalize()
    return iv + ct

def decrypt_data(encrypted_data, key):
    iv = encrypted_data[:16]
    ct = encrypted_data[16:]
    cipher = Cipher(algorithms.AES(key), modes.CFB(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    return decryptor.update(ct) + decryptor.finalize()


In [ ]:
def build_preparation_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    return Model(inputs=inp, outputs=x, name="PreparationNetwork")

def build_hiding_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    for _ in range(4):
        x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(3, (3, 3), padding='same', activation='sigmoid')(x)  # output image 3 channel
    return Model(inputs=inp, outputs=x, name="HidingNetwork")


def build_reveal_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    for _ in range(4):
        x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(3, (3, 3), padding='same', activation='sigmoid')(x)
    return Model(inputs=inp, outputs=x, name="RevealNetwork")


In [ ]:
input_shape = (64, 64, 3)

prep_net = build_preparation_network(input_shape)
hide_net = build_hiding_network((64, 64, 65 + 3))
reveal_net = build_reveal_network(input_shape)

# Encode
secret_features = prep_net(Input(shape=input_shape))
combined_input = concatenate([secret_features, Input(shape=input_shape)])
container = hide_net(combined_input)  # output: shape=(None, 64, 64, 3)

# Decode
reconstructed_secret = reveal_net(container)  # input shape cocok

# Stacked Model
encoder_input_secret = Input(shape=input_shape)
encoder_input_cover = Input(shape=input_shape)

prep_out = prep_net(encoder_input_secret)
concat = concatenate([prep_out, encoder_input_cover])
container_out = hide_net(concat)
reveal_out = reveal_net(container_out)

model = Model(inputs=[encoder_input_secret, encoder_input_cover], outputs=reveal_out)
model.compile(optimizer=Adam(), loss='mse')
model.summary()

# Training
model.fit([secret_images, cover_images], secret_images, epochs=500, batch_size=32)


In [ ]:
def sha512_hash(img_array):
    img_bytes = (img_array * 255).astype(np.uint8).tobytes()
    return hashlib.sha512(img_bytes).hexdigest()

# Contoh hashing
container_img = model.predict([secret_images[:1], cover_images[:1]])[0]
hash_value = sha512_hash(container_img)
print("SHA-512 hash of container image:", hash_value)